In [ ]:
import os
import camelot
import pandas as pd
from tqdm import tqdm
from IPython.display import display
import warnings
import re
import numpy as np
import pdfplumber

warnings.filterwarnings("ignore")

folder = os.getcwd()
pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


def extract_numbers(name):

    nums = re.findall(r'\d+', name)
    return tuple(map(int, nums))

replacements = {
    'Ⅰ': 'I',
    'Ⅱ': 'II',
    'Ⅲ': 'III',
    'Ⅳ': 'IV',
    'Ⅴ': 'V',
    'Ⅵ': 'VI',
    'Ⅶ': 'VII',
    'Ⅷ': 'VIII',
    'Ⅸ': 'IX',
    'Ⅹ': 'X'
}

def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):

    os.makedirs(out_dir, exist_ok=True)

    tables_lattice = camelot.read_pdf(
        pdf_path,
        pages=pages,
        flavor="lattice",
        line_scale=80
    )

    lattice_pages = set()
    text_pages = set()

    with pdfplumber.open(pdf_path) as pdf:

        for i, t in enumerate(tables_lattice):

            page_num = int(t.page)          
            page_idx = page_num - 1        
            lattice_pages.add(page_num)

            t.to_csv(
                os.path.join(out_dir, f"lattice_p{page_num}_{i}.csv"),
                index=False
            )

            page = pdf.pages[page_idx]

            x1, y1, x2, y2 = t._bbox
            page_w = page.width
            page_h = page.height

            table_top = page_h - y2

            left   = max(0, min(x1, page_w))
            right  = max(0, min(x2, page_w))
            top    = max(0, min(table_top - title_height, page_h))
            bottom = max(0, min(table_top, page_h))

            if right > left and bottom > top:
                title_box = (left, top, right, bottom)
                title_text = page.crop(title_box, strict=False).extract_text(
                    x_tolerance=2,
                    y_tolerance=2
                ) or ""

                if title_text.strip():
                    text_pages.add(page_num)
            else:
                title_text = ""

            with open(
                os.path.join(out_dir, f"text_p{page_num}_{i}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(title_text.strip())

    return None       


def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None

DEFAULT_MODELS_1_2_4_5 = ['ZS1414', 'ZS1212', 'ZS1012', 'ZS0812', 'ZS0808', 'ZS0608', 'ZS0607', 'ZS0407'] #для 4 и 5 нужно менять df_all
DEFAULT_MODELS3_8 = ['ZS1414', 'ZS1212', 'ZS1012', 'ZS0808', 'ZS0608', 'ZS0607', 'ZS0407']
DEFAULT_MODELS6_7 = ['ZS0508', 'ZS0610', 'ZS1216']
DEFAULT_MODELS9_10 = ['ZS1023', 'ZS1323', 'ZS1623']
DEFAULT_MODELS11 = ['ZS1623']

def extract_zs_models_checked(row):
    def extract_from_text(text):
        if pd.isna(text):
            return []
        parts = re.split(r'[;/]', str(text).upper())
        result = []
        last_zs_number = None
        
        for part in parts:
            part = part.strip()
            
            match_zs = re.search(r'ZS(\d{4})', part)

            match_digits = re.search(r'(\d{4})', part)
            
            candidate = None
            if match_zs:
                candidate = match_zs.group(0)
            elif match_digits:
                candidate = "ZS" + match_digits.group(1)
            
            if candidate and candidate in DEFAULT_MODELS11:
                result.append(candidate)
        
        return result

    models = extract_from_text(row.get("Item and Spec"))
    
    if not models:
        models = extract_from_text(row.get("Section"))

    if not models:
        return None
    
    return "; ".join(dict.fromkeys(models))

def merge_models_strict(series: pd.Series) -> str:
    if series.isna().any():
        return "; ".join(DEFAULT_MODELS11)

    uniq = []
    seen = set()
    for val in series:
        if val:
            for m in str(val).split(";"):
                m = m.strip()
                if m and m not in seen:
                    seen.add(m)
                    uniq.append(m)

    return "; ".join(uniq) if uniq else None


def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    df["Models"] = df.apply(extract_zs_models_checked, axis=1)

    group_col = None
    for col in ['Material No.', 'Код материалов', '物料编码']:
        if col in df.columns:
            group_col = col
            break
    
    if not group_col:
        return df 

    agg_map = {}
    qty_cols = ['Qty', 'Кол-во', '件数', 'Units']
    for col in qty_cols:
        if col in df.columns:
            agg_map[col] = "sum"

    text_cols = ["Item and Spec", "Remark", 'Наименование и спецификация', 
                 'Примечание', 'Версия', 'Rev.', "名称及规格", "备注", "Section"]
    for col in text_cols:
        if col in df.columns:
            agg_map[col] = join_ordered

    df_agg = df.groupby(group_col, as_index=False).agg(agg_map)
    
    models_series = df.groupby(group_col)["Models"].apply(merge_models_strict).reset_index()
    
    df_agg = df_agg.merge(models_series, on=group_col, how='left')

    return df_agg

In [ ]:
# TODO подправить Section в ZA22J-V, посчитать количества таблиц, объеденить все таблицы
pdf_files[11]

'c:\\Users\\Normc\\OneDrive\\Рабочий стол\\Python\\Работа\\Zoomlion\\ZA22J-V.pdf'

In [147]:
b = 11
e = b+1

for i, pth in enumerate(tqdm(pdf_files[b:e])):
    
    filename = os.path.basename(pth)          
    filename = os.path.splitext(filename)[0]
    print(f'Обработка файла: {filename}')

    # extract_tables_camelot(pth, filename, pages="all", title_height=800)

    dfs = []

    for file in sorted(os.listdir(filename), key=extract_numbers):

        if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
            continue

        path = os.path.join(filename, file)

        if os.path.getsize(path) == 0:  
            continue
        df_i = pd.read_csv(path, converters={1: str})

        if (
            df_i.empty
            or df_i.shape[1] < 4
            or not any(col in df_i.columns for col in ['NO.', 'Код материалов', 'No.', 'Код мат-ла', 'Serial No.', 'Part No.', '物料编码', 'п/п  Код материалов', 'Код', 'Material No.', 'Material  Code', '№  материала'])
            ):
            continue

        # если китаец насрал пробел в начале таблицы
        if 'NO.' in df_i.iloc[0].values or 'Material No.' in df_i.iloc[0].values:
            df_i.columns = df_i.iloc[0]  
            df_i = df_i[1:].reset_index(drop=True) 

        for col in ['NO.', 'No.', '№ п/п', '№ \nп/п', 'Serial No.', '序号', '№ \n\nп/п', '№ \r\nп/п', '№', 'N\r\no.', 'No\r\n.', 'Serial  number', 'НЕТ.']:
            if col in df_i.columns:
                df_i = df_i.drop(col, axis=1)

        idx = file.replace("lattice_", "").rsplit(".", 1)[0]
        section_val = os.path.join(filename, f"text_{idx}.txt")
        
        with open(section_val, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        skip_titles = [
            'Руководство по деталъная структура',
            'Альбом чертежей деталей и узлов коленчатого подъемника для высотных работ',
            'Альбом чертежей деталей и компонентов',
            'Parts Manual',
            '零部件图册',
            'Translated by Google',
            'anslated by Google'
        ]

        if lines and lines[0] in skip_titles:
            lines = lines[1:]
                
        if lines and lines[0] in ['Альбом чертежей деталей и узлов коленчатого', 'Альбом чертежей деталей и узлов коленчатого подъемника для', 'slated by Google']:
            lines = lines[2:]

        lines_joined = " ".join(lines)
            
        df_i["Section"] = lines_joined
        
        df_i = df_i[df_i.iloc[:, 0].replace('', pd.NA).notna()]
        display(df_i)
        df_i = df_i[df_i.iloc[:, 0] != '/']
        dfs.append(df_i)

    df_all = pd.concat(dfs, ignore_index=True)
    display(df_all)
    df_all.to_excel('dfs.xlsx', index=False)

    for old_col in ['во', 'Кол-\nво', 'Кол-\r\nво', 'Кол-в\nо \nштук', 'Кол-\nво \nшту\nк', 'Кол-во \nштук', 'Cantidad', 'Кол-во\n\nштук', 'Кол-во\r\nштук', 'Кол-\n\nво \n\nшту\n\nк', 'Кол-во \r\nштук', 'Кол-в\r\nо \r\nштук', 'Кол-\r\nво \r\nшту\r\nк', 'Кол-во штук', 'Кол-в', 'Колич\r\nество', 'Кол-в\r\nо\r\nштук', 'Колич ество', 'Кол-во Примечание', 'Number  of  pieces', 'Единицы', 'Единицы ']:
        if old_col in df_all.columns.tolist():
            if 'Кол-во' in df_all.columns.tolist():
                df_all['Кол-во'] = df_all.loc[:, old_col].combine_first(df_all['Кол-во'])
            else:
                df_all['Кол-во'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Кол-во' in df_all.columns:
        col = df_all.pop("Кол-во")  
        df_all.insert(2, "Кол-во", col) 

    for old_col in ['Наименование и \nспецификация', 'Наименование и \r\nспецификация', 'Наименование и\n\nспецификация', 'Nombre y especificación', 'Наименование и \r\nспецификация', 'Наименование и\r\nспецификация', 'Наименование и', 'Наименование\r\nи', 'Наименование', 'Товар и спецификация ', 'Товар  и спецификация', 'Товар  и  спецификация', 'Товар и  спецификация']:
        if old_col in df_all.columns.tolist():
            if 'Наименование и спецификация' in df_all.columns:
                df_all['Наименование и спецификация'] = df_all.loc[:, old_col].combine_first(df_all['Наименование и спецификация'])
            else:
                df_all['Наименование и спецификация'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Наименование и спецификация' in df_all.columns:
        col = df_all.pop("Наименование и спецификация")  
        df_all.insert(1, "Наименование и спецификация", col) 

    for old_col in ['Name and Spec.', 'Spec and Item', 'Name  and  specifications', 'Item  and  Spec', 'Name  and  specifications', 'Name and specifications']:
        if old_col in df_all.columns:
            if 'Item and Spec' in df_all.columns:
                df_all['Item and Spec'] = df_all.loc[:, old_col].combine_first(df_all['Item and Spec'])
            else:
                df_all['Item and Spec'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Item and Spec' in df_all.columns:
        col = df_all.pop('Item and Spec')  
        df_all.insert(1, 'Item and Spec', col) 

    for old_col in ['Примечани\nе', 'Observaciones', 'Примечани\n\nе', 'Примечани\r\nе', '№ позиции', 'Примеч\r\nание']:
        if old_col in df_all.columns:
            if 'Примечание' in df_all.columns:
                df_all['Примечание'] = df_all.loc[:, old_col].combine_first(df_all['Примечание'])
            else:
                df_all['Примечание'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Примечание' in df_all.columns:
        col = df_all.pop('Примечание')  
        df_all.insert(3, 'Примечание', col) 

    for old_col in ['Qty.', 'Unit', 'Qt\r\ny', 'Units', 'Number  of  pieces']:
        if old_col in df_all.columns:
            if 'Qty' in df_all.columns:
                df_all['Qty'] = df_all.loc[:, old_col].combine_first(df_all['Qty'])
            else:
                df_all['Qty'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Qty' in df_all.columns:
        col = df_all.pop('Qty')  
        df_all.insert(2, 'Qty', col) 
        
    for old_col in ['Código de material', 'Код мат-ла', 'п/п  Код материалов', 'Код', '№ материала	', '№ материала', '№  материала']:
        if old_col in df_all.columns:
            if 'Код материалов' in df_all.columns:
                df_all['Код материалов'] = df_all.loc[:, old_col].combine_first(df_all['Код материалов'])
            else:
                df_all['Код материалов'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Код материалов' in df_all.columns:
        col = df_all.pop('Код материалов')  
        df_all.insert(0, 'Код материалов', col) 

    for old_col in ['"Версия"', 'Версия  Кол-']:
        if old_col in df_all.columns:
            if 'Версия' in df_all.columns:
                df_all['Версия'] = df_all.loc[:, old_col].combine_first(df_all['Версия'])
            else:
                df_all['Версия'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)

    for old_col in ['Part No.', 'Material Code', 'Material  Code', 'Material  No.', 'Material  Code']:
        if old_col in df_all.columns:
            if 'Material No.' in df_all.columns:
                df_all['Material No.'] = df_all.loc[:, old_col].combine_first(df_all['Material No.'])
            else:
                df_all['Material No.'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Material No.' in df_all.columns:
        col = df_all.pop('Material No.')  
        df_all.insert(0, 'Material No.', col) 

    for old_col in ['Note']:
        if old_col in df_all.columns:
            if 'Remark' in df_all.columns:
                df_all['Remark'] = df_all.loc[:, old_col].combine_first(df_all['Remark'])
            else:
                df_all['Remark'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Remark' in df_all.columns:
        col = df_all.pop('Remark')  
        df_all.insert(3, 'Remark', col) 

    for col in ['Кол-во', 'Qty', '件数']:
        if col in df_all.columns:
            df_all[col] = df_all[col].replace({'/': 0, '、': 0, 'A': 0, '-': 0})
            df_all[col] = df_all[col].fillna(0)
            df_all[col] = df_all[col].astype(str).str.replace(',', '.', regex=False)
            df_all[col] = pd.to_numeric(df_all[col], errors='coerce').fillna(0)
            df_all[col] = df_all[col].astype(float).apply(lambda x: int(x) if x.is_integer() else x)
    
    if 'Код материалов' in df_all.columns:
        df_all['Код материалов'] = df_all['Код материалов'].astype(str).str.replace('\xa0', ' ', regex=False).str.strip()
        extracted = df_all['Код материалов'].str.extract(r'^(\S+)\s+(.*)$')
        mask = extracted[0].notna()
        df_all.loc[mask, 'Наименование и спецификация'] = extracted.loc[mask, 1]
        df_all.loc[mask, 'Код материалов'] = extracted.loc[mask, 0]

    if 'Material No.' in df_all.columns:
        df_all['Material No.'] = df_all['Material No.'].astype(str).str.replace('\xa0', ' ', regex=False).str.strip()
        extracted = df_all['Material No.'].str.extract(r'^(\S+)\s+(.*)$')
        mask = extracted[0].notna()
        df_all.loc[mask, 'Item and Spec'] = extracted.loc[mask, 1]
        df_all.loc[mask, 'Material No.'] = extracted.loc[mask, 0]

    df_all.to_excel('df_all.xlsx', index=False)
    df_all = pd.read_excel('df_all.xlsx')
    test_1 = pd.read_excel('dfs.xlsx')
    assert test_1.shape[0] == df_all.shape[0]

    for col in ['Material No.', 'Код материалов', '物料编码']:
        if col in df_all.columns:
            df_all = df_all[df_all[col] != '']
            df_all = df_all[~df_all[col].astype(str).str.startswith('/')]
            df_all = df_all[~df_all[col].isna()]

    display(df_all.head())

    if 'Material No.' in df_all.columns and df_all['Material No.'].astype(str).str.contains('NO-NUMBER').any():
        df_all['Material No.'] = df_all['Material No.'].str.split('\n').str[0]
        df_all = df_all[~df_all['Material No.'].astype(str).str.startswith('NO-NUMBER')]

    if 'Код материалов' in df_all.columns and df_all['Код материалов'].astype(str).str.contains('NO-NUMBER').any():
        df_all = df_all[~df_all['Код материалов'].astype(str).str.startswith('NO-NUMBER')]

    cols = df_all.select_dtypes(include='object').columns
    df_all[cols] = (df_all[cols].replace(r'\s*\n\s*', ' ', regex=True))
    df_all[cols] = df_all[cols].replace(replacements, regex=True)

    if 'Item and Spec' in df_all.columns:
        df_all['Item and Spec'] = df_all['Item and Spec'].replace('/', np.nan)
    
    if '物料编码' in df_all.columns:
        mask = df_all['物料编码'].str.contains(r'^\d+\s+\D', na=False)
        extracted = df_all.loc[mask, '物料编码'].str.extract(r'^(\d+)\s+(.*)$')
        df_all.loc[mask, '物料编码'] = extracted[0]
        df_all.loc[mask, '名称及规格'] = extracted[1]
        
    df_merged = merge_parts(df_all) 
   
    assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

    lst = ['ZS1623RT', 'ZA10RJE', 'ZA14J', 'ZA14JE-Li', 'ZA14NJE', 'ZA16J', 'ZA16JERT', 'ZA18J', 'ZA20J', 'ZA20JE', 'ZA20JERT', 'ZA22J-V' ,'ZA22J', 'ZA24J', 'ZA32J(H-образ.)', 'ZA32J(X-образ.)', 'ZT42J-V', 'ZA42J', '', '', '', '', '', '', '', '', '' ,'', '', 'ZT20J', 'ZT22J-V', 'ZT26J', 'ZT26JS-V', 'ZT30J', 'ZT32J-V', 'ZT32J', 'ZT34J', 'ZT38J', 'ZT42J', 'ZT51J', 'ZT58J', 'ZT72J-V', 'ZT82J', 'ZTH3507', 'ZTH3513', 'ZTH3513', 'ZX23AE']
    df_merged['Models'] = lst[b]
    df_merged

    df_merged.to_excel(f'{filename}.xlsx', index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: ZA22J-V


,Material Code,Name and specifications,Number of pieces,Remark,Section
0,00773400100261030,Plate II,1,NaN,Work platform 00773600130000000
1,1040005844,bolt,4,NaN,Work platform 00773600130000000
2,1040300771,Washer 6-200HV,32,GB/T5783-2000,Work platform 00773600130000000
3,00773400100260020,Protective cover,1,GB/T97.1-2002,Work platform 00773600130000000
4,00773400100261060,bolt,2,NaN,Work platform 00773600130000000
5,00773400100261050,Bearing,2,NaN,Work platform 00773600130000000
6,1040302480,Washer 8-200HV,4,NaN,Work platform 00773600130000000
7,1040200503,Nut M8-8,6,GB/T96.1-2002,Work platform 00773600130000000
8,1040200504,Nut M10-8,4,GB/T889.1-2000,Work platform 00773600130000000
9,1040302512,Washer 10-200HV,8,GB/T889.1-2000,Work platform 00773600130000000


,Material Code,Name and specifications,Number of pieces,Remark,Section
0,1040201111,Nut M24-10,1.0,NaN,Work platform 00773600130000000
1,1040301511,Washer 24-200HV,1.0,GB/T889-1986,Work platform 00773600130000000
2,00773400100401030,Adjustment pad,2.0,GB/T97.1-2002,Work platform 00773600130000000
3,1040006524,Bolt M10×25-10.9,8.0,NaN,Work platform 00773600130000000
4,1040004505,Bolt M24×280-10.9,1.0,NaN,Work platform 00773600130000000
5,02564400100401010,Connecting rod,2.0,GB/T5782-2000,Work platform 00773600130000000
6,1040201381,Nut M12-8,1.0,NaN,Work platform 00773600130000000
7,1040006458,Strengthened semicircular head square bolt...,1.0,GB/T6170-2000,Work platform 00773600130000000
8,1040102372,Hexagon socket head screws\r\nM6×20-A2-70,6.0,NaN,Work platform 00773600130000000
9,1999904986,Document Box,1.0,GB/T70.1-2000,Work platform 00773600130000000


,Material Code,Name and specifications,Number of pieces,Remark,Section
0,00773400100260020 Protective cover,NaN,1,NaN,Work p latform 00773600100000000
1,00773400100261030 Plate ÿ,NaN,1,NaN,Work p latform 00773600100000000
2,1040005844,bolt,4,GB/T5783-2000,Work p latform 00773600100000000
3,1040300771,Washer 6-200HV,36,GB/T97.1-2002,Work p latform 00773600100000000
4,00773400100261060 Bolt,NaN,2,NaN,Work p latform 00773600100000000
5,00773400100261050 Bearing,NaN,2,NaN,Work p latform 00773600100000000
6,1040302480,Washer 8-200HV,4,GB/T96.1-2002,Work p latform 00773600100000000
7,1040200503,Nut M8-8,7,GB/T889.1-2000,Work p latform 00773600100000000
8,1040200504,Nut M10-8,4,GB/T889.1-2000,Work p latform 00773600100000000
9,1040302512,Washer 10-200HV,8,GB/T96.1-2002,Work p latform 00773600100000000


...

,Material Code,Name and specifications,Number of pieces,Remark,Section
0,1020810958,Rocker switch,1,Emergency power and enabling,Platform electric c ontrol b ox a ssembly 0 07...
1,1020521445,Rocker switch,1,Drive orientation,Platform electric c ontrol b ox a ssembly 0 07...
2,1020521444,Rocker switch,1,lighting,Platform electric c ontrol b ox a ssembly 0 07...
3,1020521443,Rocker switch,1,Walking speed control,Platform electric c ontrol b ox a ssembly 0 07...
4,1020405061,Alarm indicator panel,1,NaN,Platform electric c ontrol b ox a ssembly 0 07...
5,00773406200211050 Platform electric control...,NaN,1,NaN,Platform electric c ontrol b ox a ssembly 0 07...
6,1020502039,Emergency stop button,1,NaN,Platform electric c ontrol b ox a ssembly 0 07...
7,1020509007,Button seat,1,NaN,Platform electric c ontrol b ox a ssembly 0 07...
8,1020521445,Rocker switch,1,Platform leveling,Platform electric c ontrol b ox a ssembly 0 07...
9,1029802085,Nylon plug,5,NaN,Platform electric c ontrol b ox a ssembly 0 07...


,Material Code,Name and specifications,Number of pieces,Remark,Section
0,1020201651,Display,1,NaN,Turntable e lectric control b ox a ssembly 0 0...
1,1020520084,Toggle switch,1,Platform rotation,Turntable e lectric control b ox a ssembly 0 0...
2,1020520084,Toggle switch,1,Platform leveling,Turntable e lectric control b ox a ssembly 0 0...
3,1020520084,Toggle switch,1,Flying boom lift,Turntable e lectric control b ox a ssembly 0 0...
4,1020520084,Toggle switch,1,Boom extension,Turntable e lectric control b ox a ssembly 0 0...
5,1020520084,Toggle switch,1,Boom lifting,Turntable e lectric control b ox a ssembly 0 0...
6,1020519658,Emergency stop button,1,Emergency stop switch,Turntable e lectric control b ox a ssembly 0 0...
7,1020509007,Button seat,1,NaN,Turntable e lectric control b ox a ssembly 0 0...
8,1020520084,Toggle switch,1,Tower boom lifting,Turntable e lectric control b ox a ssembly 0 0...
9,1020520943,Key switch,1,NaN,Turntable e lectric control b ox a ssembly 0 0...


,Material Code,Name and specifications,Number of pieces,Remark,Section,No. 1,Name and specifications
0,00773400100261030,Plate II,1,NaN,Work platform 00773600130000000,NaN,NaN
1,1040005844,bolt,4,NaN,Work platform 00773600130000000,NaN,NaN
2,1040300771,Washer 6-200HV,32,GB/T5783-2000,Work platform 00773600130000000,NaN,NaN
3,00773400100260020,Protective cover,1,GB/T97.1-2002,Work platform 00773600130000000,NaN,NaN
4,00773400100261060,bolt,2,NaN,Work platform 00773600130000000,NaN,NaN
...,...,...,...,...,...,...,...
807,1020304584,Power relay,1,Start the motor,Turntable e lectric control b ox a ssembly 0 0...,NaN,NaN
808,1020304584,Power relay,1,Radiator,Turntable e lectric control b ox a ssembly 0 0...,NaN,NaN
809,1020304584,Power relay,1,Fuel pump,Turntable e lectric control b ox a ssembly 0 0...,NaN,NaN
810,1020103903,Controller,1,NaN,Turntable e lectric control b ox a ssembly 0 0...,NaN,NaN


,Material No.,Item and Spec,Кол-во,Remark,Section,No. 1
0,00773400100261030,Plate II,1,NaN,Work platform 00773600130000000,NaN
1,1040005844,bolt,4,NaN,Work platform 00773600130000000,NaN
2,1040300771,Washer 6-200HV,32,GB/T5783-2000,Work platform 00773600130000000,NaN
3,00773400100260020,Protective cover,1,GB/T97.1-2002,Work platform 00773600130000000,NaN
4,00773400100261060,bolt,2,NaN,Work platform 00773600130000000,NaN


100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


- Спросить у Миши про запчасти, в описании которых встречается ещё одна модель, помимо основной, в которой они встречаются
- Что делать с такими в описании (Cubota): ZT138J-ZT24JE-V Turntable Decal Assembly - что значит тире?

In [ ]:
df_all = pd.read_excel('df_all.xlsx')
test_1 = pd.read_excel('dfs.xlsx')
assert test_1.shape[0] == df_all.shape[0]

for col in ['Material No.', 'Код материалов', '物料编码']:
    if col in df_all.columns:
        df_all = df_all[df_all[col] != '']
        df_all = df_all[~df_all[col].astype(str).str.startswith('/')]
        df_all = df_all[~df_all[col].isna()]

display(df_all.head())

if 'Material No.' in df_all.columns and df_all['Material No.'].astype(str).str.contains('NO-NUMBER').any():
    df_all['Material No.'] = df_all['Material No.'].str.split('\n').str[0]
    df_all = df_all[~df_all['Material No.'].astype(str).str.startswith('NO-NUMBER')]

if 'Код материалов' in df_all.columns and df_all['Код материалов'].astype(str).str.contains('NO-NUMBER').any():
    df_all = df_all[~df_all['Код материалов'].astype(str).str.startswith('NO-NUMBER')]

cols = df_all.select_dtypes(include='object').columns
df_all[cols] = (df_all[cols].replace(r'\s*\n\s*', ' ', regex=True))
df_all[cols] = df_all[cols].replace(replacements, regex=True)

if 'Item and Spec' in df_all.columns:
    df_all['Item and Spec'] = df_all['Item and Spec'].replace('/', np.nan)

if '物料编码' in df_all.columns:
    mask = df_all['物料编码'].str.contains(r'^\d+\s+\D', na=False)
    extracted = df_all.loc[mask, '物料编码'].str.extract(r'^(\d+)\s+(.*)$')
    df_all.loc[mask, '物料编码'] = extracted[0]
    df_all.loc[mask, '名称及规格'] = extracted[1]
    
df_merged = merge_parts(df_all) 

assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

lst = ['ZS1623RT', 'ZA10RJE', 'ZA14J', 'ZA14JE-Li', 'ZA14NJE', 'ZA16J', 'ZA16JERT', 'ZA18J', 'ZA20J', 'ZA20JE', 'ZA20JERT', 'ZA22J-V' ,'ZA22J', 'ZA24J', 'ZA32J(H-образ.)', 'ZA32J(X-образ.)', 'ZT42J-V', 'ZA42J', '', '', '', '', '', '', '', '', '' ,'', '', 'ZT20J', 'ZT22J-V', 'ZT26J', 'ZT26JS-V', 'ZT30J', 'ZT32J-V', 'ZT32J', 'ZT34J', 'ZT38J', 'ZT42J', 'ZT51J', 'ZT58J', 'ZT72J-V', 'ZT82J', 'ZTH3507', 'ZTH3513', 'ZTH3513', 'ZX23AE']
# df_merged['Models'] = lst[b]
df_merged

df_merged.to_excel(f'{filename}.xlsx', index=False)